# NumCompute-Stream Demo

This notebook demonstrates the main workflow for Assignment 2.2.

It uses the custom NumCompute-Stream modules to:

1. Load a CSV dataset using the custom I/O utilities.
2. Split the data into chunks to simulate streaming input.
3. Train a single decision tree incrementally using `partial_fit()`.
4. Train an ensemble tree model incrementally using `partial_fit()`.
5. Log chunk-level and cumulative accuracy with `StreamTrainer`.
6. Visualise streaming performance using the built-in `visualise.py` module.


## 1. Imports

Only NumPy and matplotlib are used, along with the custom NumCompute-Stream modules.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Make sure the project root is importable when running from the demo folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from numcompute_stream.io import load_csv, make_stream_chunks
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth,
)


## 2. Load the diabetes dataset

The dataset is loaded using the custom `load_csv()` function from `io.py`. The final column, `Outcome`, is used as the target label.


In [ ]:
data_path = PROJECT_ROOT / "demo" / "data" / "diabetes.csv"

data, columns = load_csv(str(data_path), skip_header=True)

print("Columns:", columns)
print("Data shape:", data.shape)

X = data[:, :-1]
y = data[:, -1].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class labels:", np.unique(y))


## 3. Shuffle and split into stream chunks

The rows are shuffled once, then split into multiple chunks. Each chunk simulates a new incoming batch of data.


In [ ]:
rng = np.random.default_rng(42)
indices = rng.permutation(X.shape[0])

X_stream = X[indices]
y_stream = y[indices]

chunk_size = 64
chunks = list(make_stream_chunks(X_stream, y_stream, chunk_size=chunk_size))

print("Number of chunks:", len(chunks))
print("First chunk X shape:", chunks[0][0].shape)
print("First chunk y shape:", chunks[0][1].shape)


## 4. Train a streaming decision tree

The `DecisionTreeClassifier` supports `partial_fit()`. The `StreamTrainer` handles chunk-wise fitting, scoring, and logging.


In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=5,
    criterion="gini",
    random_state=42,
)

tree_trainer = StreamTrainer(tree_model)
tree_trainer.fit_stream(chunks)

tree_history = tree_trainer.get_history()
tree_scores = tree_trainer.get_history("score")
tree_cumulative = tree_trainer.get_history("cumulative_accuracy")

print("Tree final chunk accuracy:", tree_scores[-1])
print("Tree cumulative accuracy:", tree_cumulative[-1])
print("Logged chunks:", len(tree_history))


## 5. Train a streaming ensemble model

The `EnsembleClassifier` is a Random Forest / Bagging-style model built from multiple decision trees. It also supports `partial_fit()`.


In [ ]:
ensemble_model = EnsembleClassifier(
    n_estimators=7,
    max_depth=4,
    min_samples_split=5,
    criterion="gini",
    max_features=4,
    bootstrap=True,
    random_state=42,
)

ensemble_trainer = StreamTrainer(ensemble_model)
ensemble_trainer.fit_stream(chunks)

ensemble_history = ensemble_trainer.get_history()
ensemble_scores = ensemble_trainer.get_history("score")
ensemble_cumulative = ensemble_trainer.get_history("cumulative_accuracy")

print("Ensemble final chunk accuracy:", ensemble_scores[-1])
print("Ensemble cumulative accuracy:", ensemble_cumulative[-1])
print("Logged chunks:", len(ensemble_history))


## 6. Visualise streaming accuracy

The visualisation functions are imported from the custom `visualise.py` module.


In [ ]:
fig, ax = plot_metric_over_time(
    tree_cumulative,
    title="Decision tree cumulative accuracy over chunks",
    ylabel="Cumulative accuracy",
)
plt.show()


In [ ]:
fig, ax = plot_metric_over_time(
    ensemble_cumulative,
    title="Ensemble cumulative accuracy over chunks",
    ylabel="Cumulative accuracy",
)
plt.show()


## 7. Compare tree and ensemble performance


In [ ]:
fig, ax = compare_models(
    tree_cumulative,
    ensemble_cumulative,
    labels=("Decision tree", "Ensemble"),
    title="Tree vs ensemble cumulative accuracy",
    ylabel="Cumulative accuracy",
)
plt.show()


## 8. Predictions vs ground truth on the latest chunk


In [ ]:
X_last, y_last = chunks[-1]
y_pred_last = ensemble_model.predict(X_last)

fig, ax = plot_predictions_vs_ground_truth(
    y_last,
    y_pred_last,
    title="Latest chunk: ensemble predictions vs ground truth",
)
plt.show()


## 9. Summary

This demo shows the required streaming workflow:

- The dataset is loaded using the custom I/O pipeline.
- Rows are split into chunks to simulate streaming input.
- A single decision tree and an ensemble model are trained incrementally.
- Per-chunk and cumulative metrics are logged by `StreamTrainer`.
- Streaming accuracy and prediction behaviour are visualised using `visualise.py`.
